# commit_01_data

This is the **commit_01_data** snapshot of the project: download and integrate the asx dataset.

It contains everything up to and including this milestone and nothing from later
milestones. Open the **Parameters** cell, edit if needed, then Run All — it runs
every milestone present here and finishes by running the tests.

Run with the **Python (stockxai)** kernel from this folder.

## M0 — Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists() and (ROOT.parent / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from m0_setup.config import load_config

print('environment ready')

## Parameters — edit here, then Run All

These override `config.yaml` for this notebook run. Any key you leave out keeps
the value from `config.yaml`. Changing the dates or the stock list refreshes the
raw snapshot automatically (the cache notices the change). Anything you change
flows through every stage below.

Notes:
- `n_folds × test_size` trading days are held out, oldest first.
- `explain_sample_size` caps how many test rows SHAP explains per fold.
- A full run is a few minutes; the LSTM is not part of the MVP.

In [ ]:
# ---- tunable parameters: edit here, then Run All ----
# Any key left out keeps its value from config.yaml.
PARAMS = {
    'seed': 42,
    'data': {
        'start_date': '2015-01-01',
        'end_date': '2026-09-19',
        'stocks': ['BHP', 'CBA', 'CSL', 'NAB', 'WBC', 'ANZ', 'WES', 'RIO', 'MQG', 'TLS'],
        'force_download': False,
    },
    'features': {
        'rsi_window': 14,
        'sma_windows': [10, 20, 50],
        'ema_windows': [12, 26],
        'return_lags': [1, 2, 3, 5],
    },
    'walk_forward': {'n_folds': 8, 'val_size': 60, 'test_size': 120, 'min_train_size': 500},
    'models': {
        'logistic': {'C': 1.0, 'max_iter': 1000, 'class_weight': 'balanced'},
        'xgboost': {
            'n_estimators': 300,
            'max_depth': 4,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_lambda': 1.0,
        },
    },
    'shap': {'explain_sample_size': 2000},
    'consistency': {'rolling_window': 120, 'diagnostic_window': 60, 'top_k': 10},
    'faithfulness': {'top_k': 5},
}

cfg = load_config(ROOT / 'config.yaml', overrides=PARAMS)
print('effective parameters')
print('  seed:', cfg['seed'], '| stocks:', len(cfg['data']['stocks']), '| folds:', cfg['walk_forward']['n_folds'])
print('  dates:', cfg['data']['start_date'], '->', cfg['data']['end_date'])
print('  xgboost:', cfg['models']['xgboost'])
print('  results dir:', cfg['paths']['results'])

## M1 — Data acquisition (SP1)

In [ ]:
from m1_data.data_collection import run as run_data

coverage = run_data(cfg)
coverage

## Verification

The leakage and correctness tests run as part of the milestone, not after it.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'], cwd=str(ROOT), capture_output=True, text=True
)
print(result.stdout.strip().splitlines()[-1])